In [1]:
!pip install unsloth qwen-vl-utils torchvision xformers trl peft accelerate bitsandbytes

In [ ]:
import os
import json
import random
import torch
from PIL import Image
from datasets import Dataset
from unsloth import FastVisionModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from transformers import AutoProcessor

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
IMAGES_FOLDER = 'images'
TEXTS_FOLDER = 'texts'
OUTPUT_FILE = 'dataset.json'
instructions = [
    'Analyze this Intraday Volume chart. Provide pattern, strategy, and volatility.',
    'วิเคราะห์กราฟ Options นี้โดยละเอียด',
    'Explain the Put/Call distribution and suggest trading strategies.',
    'อธิบายกราฟและให้คำแนะนำการเทรด',
]
USER_INSTRUCTION = random.choice(instructions)

In [4]:
dataset = []
try:
  image_files = os.listdir(IMAGES_FOLDER)
  print(f"Found {len(image_files)} image files.")

  for image in image_files:

    base_name = os.path.splitext(image)[0]
    text_file = base_name+'.txt'
    text_path = os.path.join(TEXTS_FOLDER, text_file)

    if os.path.exists(text_path):
      with open(text_path, 'r', encoding='utf-8') as f:
        analysis = f.read().strip()
      dataset.append({
          'image_path': os.path.join(os.getcwd(), IMAGES_FOLDER, image),
          'user_input': USER_INSTRUCTION,
          'model_output': analysis
      })
    else:
      print(f'warning: missing text for {image}')

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
      json.dump(dataset, f, indent=2, ensure_ascii=False)
except Exception as e:
  print(f"An error occurred: {e}")
  raise

Found 97 image files.


In [5]:
BASE_MODEL = 'unsloth/Qwen3-VL-4B-Instruct'
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUMULATION = 4
LEARNING_RATE = 2e-4
EPOCHS = 5

In [6]:
model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    trust_remote_code=True,
    use_gradient_checkpointing=True
)

processor = AutoProcessor.from_pretrained(BASE_MODEL)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Qwen3_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.278 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_Vl does not support SDPA - switching to fast eager.


In [7]:
model = FastVisionModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [8]:
with open("dataset.json", "r", encoding="utf-8") as f:
        data = json.load(f)
dataset = Dataset.from_list(data)
print(f"Dataset size: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")

Dataset size: 96
Dataset columns: ['image_path', 'user_input', 'model_output']


In [9]:
def convert_to_conversation_format(example):
    """Convert each example to conversation messages format"""
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example['image_path']},
                {"type": "text", "text": example['user_input']},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example['model_output']}
            ],
        },
    ]
    return {"messages": conversation}

In [10]:
formatted_data = [convert_to_conversation_format(ex) for ex in data]

In [11]:
dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")

Dataset size: 96
Dataset columns: ['messages']


In [12]:
if len(dataset) > 10:
    dataset = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = dataset['train']
    eval_dataset = dataset['test']
    print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
else:
    train_dataset = dataset
    eval_dataset = None
    print("Dataset too small for split, using all for training")

Train: 86, Eval: 10


In [ ]:
print("Setting up trainer...")
from unsloth.trainer import UnslothVisionDataCollator
data_collator = UnslothVisionDataCollator(model, tokenizer)

Setting up trainer...
Unsloth: Model does not have a default image size - using 512


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1, 
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="epoch" if eval_dataset else "no",
        save_strategy="epoch",
        output_dir="outputs",
        optim="adamw_8bit",
        seed=3407,
        report_to="none",
        save_total_limit=2,
    ),
)


In [15]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss
1,2.359300,0.944334
2,1.451900,0.835829
3,1.404400,0.805158
4,1.039600,0.808267
5,1.037900,0.824295


Unsloth: Not an error, but Qwen3VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [16]:
print("Saving model...")
model.save_pretrained("oi_analyst_model")
tokenizer.save_pretrained("oi_analyst_model")

print("\n=== Training Complete ===")
print(f"Total training time: {trainer_stats.metrics.get('train_runtime', 0):.2f}s")
print(f"Final loss: {trainer_stats.metrics.get('train_loss', 0):.4f}")
print(f"Model saved to: oi_analyst_model/")

Saving model...

=== Training Complete ===
Total training time: 673.88s
Final loss: 2.9768
Model saved to: oi_analyst_model/


In [ ]:
print("\n=== Testing Model ===")
print("Loading model...")
model, tokenizer = FastVisionModel.from_pretrained(
    "oi_analyst_model",
    load_in_4bit=True,
    trust_remote_code=True,
)

# Switch to inference mode (disables dropout, etc.)
FastVisionModel.for_inference(model)

# Load processor
processor = AutoProcessor.from_pretrained("oi_analyst_model")

print("Model loaded successfully!")


=== Testing Model ===
Loading model...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Qwen3_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.278 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_Vl does not support SDPA - switching to fast eager.
Model loaded successfully!


In [ ]:
def analyze_chart(image_path, instruction="Analyze this Intraday Volume chart."):
    """
    Generate chart analysis from an image
    
    Args:
        image_path: Path to the chart image
        instruction: User instruction/prompt
    
    Returns:
        Generated analysis text
    """
    
    # Load the actual image file
    try:
        image = Image.open(image_path).convert('RGB')
        print(f"✓ Loaded image: {image.size}")
    except Exception as e:
        return f"Error loading image: {e}"
    
    # Create conversation messages
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": instruction}
            ]
        }
    ]
    
    # Apply chat template (creates formatted text with <image> tokens)
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    print(f"✓ Formatted prompt (first 100 chars): {text[:100]}...")
    
    # Process text AND image together
    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True
    ).to(model.device)
    
    print(f"✓ Input shape: {inputs['input_ids'].shape}")
    
    # Generate
    print("Generating response...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,     
            temperature=0.7,          
            do_sample=True,           
            top_p=0.9,               
            repetition_penalty=1.1, 
        )
    
    # Decode output
    full_response = processor.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response (remove the prompt)
    if "<|im_start|>assistant" in full_response:
        # Split at assistant token and take the last part
        response = full_response.split("<|im_start|>assistant")[-1]
        response = response.replace("<|im_end|>", "").strip()
    else:
        # Fallback: remove the input text
        response = full_response.replace(text, "").strip()
    
    return response

In [19]:
test_image_path = "test_oi_oil.jpg"
print("\n" + "="*60)
print("Testing inference...")
print("="*60 + "\n")
    
# Generate analysis
analysis = analyze_chart(
        test_image_path,
        instruction="วิเคราะห์กราฟ Options นี้โดยละเอียด"
    )
    
print("\n" + "="*60)
print("GENERATED ANALYSIS:")
print("="*60)
print(analysis)
print("="*60 + "\n")



Testing inference...

✓ Loaded image: (1141, 881)
✓ Formatted prompt (first 100 chars): <|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>วิเคราะห์กราฟ Options นี้โดยละเอียด<|im_...
✓ Input shape: torch.Size([1, 1035])
Generating response...

GENERATED ANALYSIS:
user
วิเคราะห์กราฟ Options นี้โดยละเอียด
assistant
📊วิเคราะห์กราฟ WTI Crude Oil (LO4Z5) Intraday Volume: สามารถสรุปรายละเอียดได้ดังนี้

1. ภาพรวมการเคลื่อนไหว (The Pattern)

▪️Volatility Skew (เส้นประสีชมพู): เส้น Vol Settle มีลักษณะเป็นรูปตัว U ในบริเวณราคาใช้สิทธิต่ำกว่า 58.5 และมีค่าสูงขึ้นอย่างชัดเจนครับๆ เมื่อราคาใช้สิทธิสูงขึ้นไปทางขวา สังเกตได้ว่าความผันผวนจะลดลงในฝั่งซ้ายและสูงขึ้นในฝั่งขวา โดยเฉพาะที่ Strike Price 59.25 ที่มีค่า Volatility ใกล้เคียงกับ 20.00 - 22.40

▪️Volume Distribution (แท่งกราฟสีส้มและน้ำเงิน):

🟠Put Volume (สีส้ม): มีปริมาณรวมสูงกว่า Call Volume โดยเฉพาะอย่างยิ่งที่ระดับราคาใช้สิทธิที่ต่ำ เช่น:

Strike 57.5AP: มี Put Volume สูงมากถึงประมาณ 50-60 (แท่งสีส้มโดดเด่นที่สุดในกราฟ)
Strike 58.

In [20]:
test_image_path = "test_oi_gold.jpg"
print("\n" + "="*60)
print("Testing inference...")
print("="*60 + "\n")
    
# Generate analysis
analysis = analyze_chart(
        test_image_path,
        instruction='Analyze this Intraday Volume chart. Provide pattern, strategy, and volatility.'
    )
    
print("\n" + "="*60)
print("GENERATED ANALYSIS:")
print("="*60)
print(analysis)
print("="*60 + "\n")



Testing inference...

✓ Loaded image: (1058, 794)
✓ Formatted prompt (first 100 chars): <|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Analyze this Intraday Volume chart. Prov...
✓ Input shape: torch.Size([1, 852])
Generating response...

GENERATED ANALYSIS:
user
Analyze this Intraday Volume chart. Provide pattern, strategy, and volatility.
assistant
Based on the provided Gold (OG|GC) G5MZ5 Intraday Volume chart, here's a breakdown of the patterns observed:

1.  📊 Market Sentiment & Pattern

The volume distribution suggests a bearish to neutral sentiment in the current session.

▪️Put Dominance: There are significantly more Put contracts traded than Call contracts.
Volume Breakdown:
Put: 1,270
Call: 1,033
This indicates that market participants have been actively buying or selling Puts, likely as a hedge against downside risk or for speculation.

▪️Activity Clusters:
▪️🟠High Put Activity: The highest put volumes cluster around strikes $4500 and slightly lower ($4487-$4500